# 📊 Predicción de Cancelación de Clientes - Telecom X
Este notebook implementa un flujo completo de **Machine Learning** para predecir la cancelación de clientes.

Incluye:
- Exploración de datos (correlación y visualización)
- Preprocesamiento
- Modelado con **Regresión Logística, KNN, Random Forest y SVM**
- Evaluación con métricas
- Análisis de importancia de variables
- Conclusiones estratégicas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42


## 1. Cargar datos y preparar

In [ ]:
# Subir archivo en Colab: usa la interfaz de Colab (Archivos -> Subir)
# Asegúrate de renombrar tu archivo a 'telecom_clientes.csv' o ajusta el nombre abajo

df = pd.read_csv("telecom_clientes.csv")

# Eliminar columnas irrelevantes si existen
df = df.drop(columns=[c for c in ["customerID", "CustomerID", "id"] if c in df.columns], errors="ignore")

# Convertir Churn a binario (0/1)
if df["Churn"].dtype == "object":
    df["Churn"] = df["Churn"].str.lower().map({"yes":1, "si":1, "y":1, "1":1, "true":1, "no":0, "n":0, "0":0, "false":0})

df.head()

## 2. Matriz de correlación

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), cmap="coolwarm", center=0)
plt.title("Matriz de correlación")
plt.show()

df.corr()["Churn"].sort_values(ascending=False).head(10)

## 3. Variables clave vs Cancelación

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.boxplot(data=df, x="Churn", y="tenure", ax=ax[0])
sns.boxplot(data=df, x="Churn", y="TotalCharges", ax=ax[1])
ax[0].set_title("Tenure vs Churn")
ax[1].set_title("TotalCharges vs Churn")
plt.show()

## 4. División en train/test

In [ ]:
X = df.drop(columns=["Churn"])
y = df["Churn"]

num_cols = X.select_dtypes(include=[np.number]).columns
cat_cols = X.select_dtypes(exclude=[np.number]).columns

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=RANDOM_STATE)

## 5. Modelos

In [ ]:
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

preprocess = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", ohe, cat_cols)
])

scaler = StandardScaler(with_mean=False)

models = {
    "LogisticRegression": Pipeline([("prep", preprocess), ("scale", scaler), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "KNN": Pipeline([("prep", preprocess), ("scale", scaler), ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance"))]),
    "RandomForest": Pipeline([("prep", preprocess), ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced_subsample", random_state=RANDOM_STATE))]),
    "LinearSVM": Pipeline([("prep", preprocess), ("scale", scaler), ("clf", LinearSVC(class_weight="balanced", random_state=RANDOM_STATE))])
}

def evaluar(nombre, modelo):
    modelo.fit(X_train, y_train)
    y_pred = modelo.predict(X_test)
    print(f"\n▶ {nombre}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precisión:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1-score:", f1_score(y_test, y_pred))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"Matriz de confusión - {nombre}")
    plt.show()
    print(classification_report(y_test, y_pred))
    return modelo

fitted = {}
for name, model in models.items():
    fitted[name] = evaluar(name, model)

## 6. Importancia de variables

In [ ]:
# Obtener nombres de features tras OHE
def get_feature_names(prep, X_fit):
    prep.fit(X_fit)
    num_names = list(num_cols)
    cat_names = list(prep.named_transformers_["cat"].get_feature_names_out(cat_cols))
    return num_names + list(cat_names)

feat_names = get_feature_names(preprocess, X_train)

# Ejemplo: importancia en Random Forest
rf = fitted["RandomForest"].named_steps["clf"]
imp_rf = pd.Series(rf.feature_importances_, index=feat_names).sort_values(ascending=False)
imp_rf.head(15).plot(kind="barh", figsize=(8,6))
plt.title("Importancia de variables - Random Forest")
plt.show()

## 7. Conclusiones estratégicas
- Clientes con **tenure bajo** tienden a cancelar más.
- Contratos **mensuales** están más asociados a churn.
- **TotalCharges bajo** (clientes recientes) se relaciona con mayor churn.
- Ausencia de servicios extra como **TechSupport/OnlineSecurity** aumenta la probabilidad de cancelación.
- Estrategias: onboarding reforzado, incentivos a contratos largos, bundles de servicios, gestión de precios y métodos de pago.